# 01. Framing & ingestion

This notebook is a project setup and scope definition: the operational problem and constraints, the data and operational metrics and benchmarks. It then loads the raw AIS data. 


## Contents

1. Problem framing
2. Operational constraints
3. Metrics
4. Operational reference (IAMSAR)
5. Setup & ingestion


## 1. Problem framing

#### Goals

A vessel stops reporting. The last available signal is its AIS track. We want to predict where it is, or will be, at a chosen horizon, so the area to search is smaller than "somewhere along its possible routes".

Knowing a vessel's likely position matters beyond a distress search. A vessel that stops broadcasting AIS may simply have a technical problem while still moving normally, and tracking where it probably is stays useful for safety and situational awareness in that case too.

Rather than a classic drift-based SAR, the goal here is to predict a powered vessel that follows routes and intent, using only its own kinematics: position, speed and heading over time. 


#### Terms we'll use
- **prediction horizon**: the time window in the future at the end of which we want to predict the position of the vessels: prediction horizon 3h = "where will the vessel be 3h from now". 
- **lookback window**: the available AIS history up to present that we can use to predict the future. lookback 12h = the AIS recording for the past 12h. 
- **lag features** and **nb lags**: number of past position available for the prediction: if the lookback window is 12h with one recording per hour, there are 12 lag features (12 lag positions) available. 
- **haversine distance**: the great circle distance from point A to point B, in order to account for Earth curvature. 
- **uncertainty radius R(p,h)**: at horizon h, p% of prediction errors are lower than R. 


## 2. Operational constraints

The idea is to use a simple, minimal workflow to simulate a situation where a robust, lean and simple system is required:

- AIS only. No weather, no bathymetry, no external destination or route data.
- The user must be able to ask any prediction horizon and get a prediction with a level of uncertainty, without having to rerun a dataset preprocessing.
- In particular, no tuning of the lookback window to the prediction horizon.
- One model for all horizons, not one model per horizon (impracticable to maintain and to serve).
- Simplicity first. Low compute, low latency and few moving parts take priority over squeezing the last kilometer of accuracy.

If we had additional information like the destination intent of the vessel (most of which follow pre-established commercial routes) we would be able to rapidly handle easy prediction with a simple geometrical extrapolation (a vessel heading straight to its destination will probably keep doing so). However, the AIS destination field is free text, hand-entered, often stale or empty. So we'll process all the vessels in the same way.

Without a known or assumable route, projecting the vessel's own recent motion forward is close to the only usable estimate available. This is why the constant-velocity extrapolation used later as the baseline carries real operational weight: it is close to the best a purely kinematic approach can offer once no route information is available.


## 3. Metrics

We use two different metrics: 
- On a model level, we use the haversine MAE (km). The mean great-circle distance between predicted and true position, used to rank models across notebooks. 

- On an operational level, we use the uncertainty radius R(p,h). It is the p-th percentile of the haversine error distribution for predictions at horizon h. R(90, 12) = 100 means: "for 12h-horizon predictions 90% of prediction error (haversine distance between prediction position and true position) are below 100km". 

*Important note*: R90 is a **descriptive** statement about past errors: 90% of predictions at a given horizon landed within R(90,h) km of the truth. It is not by itself a 90% probability for a new vessel. It's possible to run a calibration check on the held-out test set to see whether close to 90% of true positions fall inside the R90 disk built on training data.

**Limitation:** R(p,h) is a single fleet-wide radius: the p-th percentile of all errors at horizon h, pooled across vessels. It is not conditioned on the vessel's state, so the same disk is applied to every prediction, even though a stationary vessel typically needs a much smaller radius than a maneuvering one.


## 4. Operational reference (IAMSAR)

We do not pre-establish an accuracy target but we'll use a horizon-independent benchmark from maritime SAR practice.

We use the 10 NM line as a fixed reference on the radius-versus-horizon plots. Source: IAMSAR Manual, Volume III (Mobile Facilities), Section 3, On-scene co-ordination, "Search area". Published by ICAO/IMO (ICAO Doc 9731). Default search radius R = 10 NM when the search must commence immediately.



## 5. Setup & ingestion


In [1]:
import warnings
from datetime import datetime

warnings.filterwarnings("ignore")

import pandas as pd

from vessel_tracker.config import settings
from vessel_tracker.paths import DATA_RAW
from vessel_tracker.preprocessing import download_source_files

print(f"Ingestion period: {settings.ingestion.start_date} → {settings.ingestion.end_date}")


Ingestion period: 2024-11-01 → 2024-11-30


### 5.1 Ingestion

NOAA download (optional if raw is already present). Filters bbox, vessel type (cargo/tanker, 70-89), and useful columns. See `download_source_files()`.


In [2]:
# Uncomment if raw does not exist yet (slow, downloads day by day from NOAA).
# ing = settings.ingestion
# bb = ing.bounding_box
# download_source_files(
#     start_date=ing.start_date,
#     end_date=ing.end_date,
#     lon_west=bb.lon_west,
#     lon_east=bb.lon_east,
#     lat_north=bb.lat_north,
#     lat_south=bb.lat_south,
#     output_path=str(DATA_RAW.parent),
# )


### 5.2 Raw load

File: `data/raw/AIS_merged_{coords}_{dates}.parquet`.
Geographic and vessel-type filtering is done at ingestion; `VesselType` is no longer present.


In [3]:
ing = settings.ingestion
bb = ing.bounding_box
coords = f"lon{bb.lon_west:.1f}to{bb.lon_east:.1f}_lat{bb.lat_south:.1f}to{bb.lat_north:.1f}"
start = datetime.strptime(ing.start_date, "%Y-%m-%d").strftime("%Y%m%d")
end = datetime.strptime(ing.end_date, "%Y-%m-%d").strftime("%Y%m%d")
raw_path = DATA_RAW / f"AIS_merged_{coords}_{start}to{end}.parquet"

if not raw_path.exists():
    raise FileNotFoundError(
        f"No file 'AIS_merged_{coords}_{start}to{end}.parquet' in {DATA_RAW}. "
        "Run ingestion (cell above) or scripts/preprocess.py."
    )

df = pd.read_parquet(raw_path)
print(f"Loaded {raw_path.name}: {len(df):,} rows, {df['MMSI'].nunique():,} vessels")

Loaded AIS_merged_lon-95.0to-77.2_lat22.5to29.5_20241101to20241130.parquet: 7,532,915 rows, 2,441 vessels


### 5.3 Raw preview

Structure and orders of magnitude **before** cleaning, resampling, or split (continued in notebook 02).


In [4]:
print("Shape:", df.shape)
print("\nColumns:", list(df.columns))
df.dtypes


Shape: (7532915, 11)

Columns: ['MMSI', 'BaseDateTime', 'LAT', 'LON', 'SOG', 'COG', 'Heading', 'Status', 'Length', 'Width', 'Draft']


MMSI              int64
BaseDateTime     object
LAT             float64
LON             float64
SOG             float64
COG             float64
Heading         float64
Status          float64
Length          float64
Width           float64
Draft           float64
dtype: object

In [5]:
df.head()

,MMSI,BaseDateTime,LAT,LON,SOG,COG,Heading,Status,Length,Width,Draft
0,636020921,2024-11-01T00:00:02,28.94677,-89.40084,10.8,211.4,208.0,0.0,182.0,32.0,11.5
1,372101000,2024-11-01T00:00:01,29.11136,-89.26917,12.9,25.0,26.0,0.0,183.0,32.0,8.0
2,368189190,2024-11-01T00:00:02,29.35678,-90.25063,0.0,90.0,337.0,0.0,9.0,0.0,3.9
3,636021410,2024-11-01T00:00:07,29.37746,-94.86622,6.3,284.7,284.0,0.0,124.0,20.0,8.7
4,352306000,2024-11-01T00:00:07,26.08509,-80.11435,0.0,206.8,6.0,0.0,64.0,14.0,3.0


In [6]:
df.describe(include="all")

,MMSI,BaseDateTime,LAT,LON,SOG,COG,Heading,Status,Length,Width,Draft
count,7.532915e+06,7532915,7.532915e+06,7.532915e+06,7.532915e+06,7.532915e+06,7.532915e+06,7.520510e+06,7.495378e+06,7.423868e+06,7.521319e+06
unique,NaN,2325757,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,2024-11-19T19:00:04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,36,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,4.149121e+08,NaN,2.788859e+01,-8.844105e+01,5.960109e+00,1.894508e+02,2.072760e+02,9.990495e-01,1.620087e+02,2.737625e+01,7.321785e+00
std,1.291641e+08,NaN,1.695314e+00,5.699606e+00,6.305038e+00,1.067619e+02,1.424838e+02,2.009188e+00,8.054123e+01,1.172949e+01,3.557651e+00
min,8.000000e+00,NaN,2.250717e+01,-9.503697e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,3.382930e+08,NaN,2.652947e+01,-9.407607e+01,0.000000e+00,1.016000e+02,9.100000e+01,0.000000e+00,8.800000e+01,1.800000e+01,4.400000e+00
50%,3.676776e+08,NaN,2.877213e+01,-9.019975e+01,3.100000e+00,1.887000e+02,1.800000e+02,0.000000e+00,1.800000e+02,3.000000e+01,7.700000e+00
75%,5.380080e+08,NaN,2.915008e+01,-8.240342e+01,1.180000e+01,2.825000e+02,3.060000e+02,1.000000e+00,2.250000e+02,3.200000e+01,9.700000e+00


## 6. Conclusion

- **Scope fixed:** AIS-only, cargo/tanker (70-89), Gulf bounding box, Nov 2024; raw load **7,532,915** rows / **2,441** vessels.
- **Metrics defined:** haversine MAE for model comparison, R(p,h) for the operational output; external reference IAMSAR **10 NM**.
- **Next:** cleaning, resampling, split and EDA (NB02).
